# Data profiling and metric discovery
Builds on `01_api_exploration.ipynb`. Fetches **all available history**, without a `received_since` filter, from drives and warnings. Pages are cached under ignored `data/profiling/`; interrupted runs resume, and completed reruns make no API calls. The earlier 28-day extract is preserved in `data/profiling_28day_backup/`.

`scope.json` fixes the receipt cutoff for reproducible analysis. Raw JSON caches retain every fetched record; analysis excludes receipts after that cutoff. This live API has no documented snapshot guarantee, so concurrent changes and backfills can still affect traversal. Counts can differ from the earlier counting notebook because the API continues receiving events.

The 2,000-page safety guard remains; the summary explicitly reports whether both endpoints reached their final page. To request a fresh extract, archive the cache first. Credentials and cursors are never displayed.

Latest refresh: 27 October 2026, using `received_since` with a 24-hour overlap from each endpoint's latest cached receipt. Exact payloads already cached are removed from the incoming overlap; reports with different receipt metadata are preserved. The cutoff is the refresh start time, not the end of the day. Backdated receipts outside the overlap and source deletions are not guaranteed to be captured. A pre-refresh backup is retained under `data/profiling_backup_*/`. Normal notebook reruns continue to use saved data; they do not trigger a fresh update.

In [ ]:
from pathlib import Path
import json
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv

ROOT = Path.cwd()
load_dotenv(ROOT / ".env")
BASE_URL = "https://qw7ytrkpd3swtndax6c4a73oua0mcrky.lambda-url.eu-west-1.on.aws"
CACHE = ROOT / "data" / "profiling"
CACHE.mkdir(parents=True, exist_ok=True)
PAGE_SIZE, MAX_PAGES = 1000, 2000
config_path = CACHE / "scope.json"
if config_path.exists():
    scope = json.loads(config_path.read_text())
else:
    as_of = pd.Timestamp.now(tz="UTC")
    scope = {"as_of": as_of.isoformat(), "history": "all_available", "received_since": None}
    config_path.write_text(json.dumps(scope))
if scope.get("history") != "all_available" or scope.get("received_since") is not None:
    raise RuntimeError("This notebook requires an all-history cache; archive the old filtered cache first.")
display(pd.DataFrame([scope]))
pd.set_option("display.max_columns", 15)
pd.set_option("display.max_rows", 30)

## Cached pagination
Authenticate only when another page is needed. GET retries are limited to transient errors, with server Retry-After respected when numeric. Each successful page is checkpointed; cursors remain local. No unbounded pagination or background refresh.

In [ ]:
session = requests.Session()
authenticated = False

def authenticate():
    global authenticated
    credentials = {"grant_type": "client_credentials", "client_id": os.getenv("OOONO_CLIENT_ID"), "client_secret": os.getenv("OOONO_CLIENT_SECRET")}
    if not credentials["client_id"] or not credentials["client_secret"]:
        raise RuntimeError("Configure local .env credentials.")
    response = session.post(f"{BASE_URL}/oauth/token", data=credentials, timeout=(10, 60))
    if response.status_code != 200:
        raise RuntimeError(f"Authentication HTTP {response.status_code}; body hidden.")
    session.headers.update({"Authorization": "Bearer " + response.json()["access_token"]})
    authenticated = True

def get_page(endpoint, params):
    if not authenticated:
        authenticate()
    for attempt in range(4):
        try:
            response = session.get(f"{BASE_URL}/v1/{endpoint}", params=params, timeout=(10, 60))
        except (requests.Timeout, requests.ConnectionError):
            if attempt == 3:
                raise RuntimeError(f"{endpoint}: request failed; cached pages retained.") from None
            time.sleep(2 ** attempt)
            continue
        if response.status_code == 401 and attempt < 3:
            authenticate()
            continue
        if response.status_code in [429, 500, 502, 503, 504] and attempt < 3:
            wait = response.headers.get("Retry-After", "")
            time.sleep(float(wait) if wait.isdigit() else 2 ** attempt)
            continue
        if response.status_code != 200:
            raise RuntimeError(f"{endpoint}: HTTP {response.status_code}; body hidden.")
        return response.json()
    raise RuntimeError("Retry budget exhausted.")

def load_endpoint(endpoint):
    path = CACHE / f"{endpoint}.json"
    state = json.loads(path.read_text()) if path.exists() else {"records": [], "cursor": None, "pages": 0, "complete": False, "seen_cursors": []}
    calls = 0
    while not state["complete"] and state["pages"] < MAX_PAGES:
        params = {"limit": PAGE_SIZE}
        if state["cursor"]:
            params["cursor"] = state["cursor"]
        payload = get_page(endpoint, params)
        list_keys = [k for k, v in payload.items() if isinstance(v, list)]
        if len(list_keys) != 1 or "next_cursor" not in payload:
            raise RuntimeError("Unexpected page envelope; inspect structure before continuing.")
        cursor = payload["next_cursor"]
        if cursor and cursor in state["seen_cursors"]:
            raise RuntimeError("Repeated pagination cursor; stopping to avoid a loop.")
        state["records"].extend(payload[list_keys[0]])
        state.update(cursor=cursor, pages=state["pages"] + 1, complete=not bool(cursor))
        if cursor:
            state["seen_cursors"].append(cursor)
        temp = path.with_suffix(".tmp")
        temp.write_text(json.dumps(state))
        temp.replace(path)
        calls += 1
    frame = pd.json_normalize(state["records"])
    # The API has no documented upper receipt bound; freeze it locally.
    receipt = pd.to_datetime(frame["received_at"], utc=True, errors="coerce")
    keep = receipt.isna() | (receipt <= pd.Timestamp(scope["as_of"]))
    summary = {"endpoint": endpoint, "pages": state["pages"], "new_calls": calls, "pagination_exhausted": state["complete"], "fetched_rows": len(frame), "after_as_of_excluded": int((~keep).sum()), "profile_rows": int(keep.sum()), "receipt_min": receipt.min(), "receipt_max": receipt.max()}
    summary["last_refresh_pages"] = state.get("last_refresh", {}).get("pages", 0)
    summary["last_refresh_added_rows"] = state.get("last_refresh", {}).get("added_rows", 0)
    return frame.loc[keep].copy(), summary

drives, drive_extract = load_endpoint("drives")
warnings, warning_extract = load_endpoint("warnings")
display(pd.DataFrame([drive_extract, warning_extract]))

## Identity, nulls, and distributions
Duplicate rows and repeated IDs are separate checks. Conflicting IDs are excluded from relationship checks rather than silently choosing a winner. Null action means “no action value recorded”; it is not proof of an ignored warning.

Raw-row distributions below intentionally retain repeats for discovery and are not metric denominators. The conflicting-ID check counts any differing payload, including receipt metadata; this does not yet establish a business-field conflict.

In [ ]:
def identity(df, key):
    distinct = df.drop_duplicates()
    conflicts = distinct.loc[distinct[key].notna(), key].duplicated(keep=False)
    return {"rows": len(df), "null_ids": int(df[key].isna().sum()), "distinct_ids": df[key].nunique(), "extra_rows_on_repeated_ids": int(df[key].dropna().duplicated().sum()), "exact_duplicate_rows": int(df.duplicated().sum()), "conflicting_ids": distinct.loc[distinct[key].notna()].loc[conflicts, key].nunique(), "business_conflicting_ids": df.drop(columns=["received_at"], errors="ignore").drop_duplicates().loc[lambda x: x[key].notna() & x[key].duplicated(keep=False), key].nunique()}

display(pd.DataFrame({"drives": identity(drives, "session_id"), "warnings": identity(warnings, "warning_id")} ).T)
nulls = pd.concat([df.isna().sum().rename(name) for name, df in [("drives", drives), ("warnings", warnings)]], axis=1)
display(nulls.rename_axis("column").fillna("—"))

def frequencies(df, column):
    counts = df[column].value_counts(dropna=False)
    return pd.DataFrame({"rows": counts, "pct": (counts / len(df) * 100).round(2)})

for col in ["platform", "app_version", "connection_count"]:
    print(f"Drives: {col}")
    display(frequencies(drives, col))
for col in ["pin_type", "user_action"]:
    print(f"Warnings: {col}")
    display(frequencies(warnings, col))

numeric = drives[["duration_s", "connection_count"]].apply(pd.to_numeric, errors="coerce")
display(numeric.describe(percentiles=[.01, .05, .5, .9, .95, .99]).round(2))
display(drives.groupby(["platform", "app_version"], dropna=False).agg(rows=("session_id", "size"), users=("user_id", "nunique"), median_duration_s=("duration_s", "median"), median_connections=("connection_count", "median")))
# Which fields vary within repeated sessions? Distinguish replay metadata from business changes.
repeated = drives.loc[drives["session_id"].duplicated(keep=False)]
if len(repeated):
    differing = repeated.groupby("session_id").nunique(dropna=False).gt(1).sum()
    print("Fields differing within repeated session IDs")
    display(differing[differing.gt(0)].rename("session_ids_with_differences").to_frame())


## Timestamp consistency
`received_at` is explicitly UTC. Device start/end and warning timestamps may lack offsets. Preserve naive device times: do not assume they are UTC or infer a country timezone. Same-clock duration checks are possible; device-to-receipt differences below are **wall-clock diagnostics, not delivery latency**. One-second duration differences may be rounding.

In [ ]:
def parse_device(series):
    # Profile source strings before parsing; the observed sample is offset-free.
    aware = series.dropna().astype(str).str.contains(r"(?:Z|[+-]\d{2}:?\d{2})$", regex=True)
    if aware.any():
        raise RuntimeError("Offset-bearing device timestamps found; separate timezone groups before comparison.")
    return pd.to_datetime(series, format="mixed", errors="coerce")

start = parse_device(drives["started_at"])
end = parse_device(drives["ended_at"])
warning_time = parse_device(warnings["device_time"])
received = pd.to_datetime(drives["received_at"], utc=True, errors="coerce")
warning_received = pd.to_datetime(warnings["received_at"], utc=True, errors="coerce")
elapsed = (end - start).dt.total_seconds()
duration_delta = numeric["duration_s"] - elapsed
checks = {
    "unparseable nonnull drive start": (drives["started_at"].notna() & start.isna()).sum(),
    "unparseable nonnull drive end": (drives["ended_at"].notna() & end.isna()).sum(),
    "unparseable nonnull drive receipt": (drives["received_at"].notna() & received.isna()).sum(),
    "unparseable nonnull warning time": (warnings["device_time"].notna() & warning_time.isna()).sum(),
    "unparseable nonnull warning receipt": (warnings["received_at"].notna() & warning_received.isna()).sum(),
    "warning device date > extract date + 2 days (clock diagnostic)": (warning_time > pd.Timestamp(scope["as_of"]).tz_localize(None) + pd.Timedelta(days=2)).sum(),
    "drive start date > extract date + 2 days (clock diagnostic)": (start > pd.Timestamp(scope["as_of"]).tz_localize(None) + pd.Timedelta(days=2)).sum(),
    "end before start": (elapsed < 0).sum(),
    "negative duration": (numeric["duration_s"] < 0).sum(),
    "zero duration": (numeric["duration_s"] == 0).sum(),
    "duration mismatch > 1 second": (duration_delta.abs() > 1).sum(),
    "negative connection count": (numeric["connection_count"] < 0).sum(),
    "fractional connection count": (numeric["connection_count"].dropna() % 1 != 0).sum(),
    "nonnumeric duration": (drives["duration_s"].notna() & numeric["duration_s"].isna()).sum(),
    "nonnumeric connection count": (drives["connection_count"].notna() & numeric["connection_count"].isna()).sum(),
}
display(pd.Series(checks, name="rows_flagged").to_frame())
display(pd.DataFrame({"start_wall_time": [start.min(), start.max()], "end_wall_time": [end.min(), end.max()], "receipt_utc": [received.min(), received.max()], "warning_wall_time": [warning_time.min(), warning_time.max()]}, index=["min", "max"]))
display(pd.DataFrame({"duration_minus_elapsed_s": duration_delta, "receipt_minus_end_wall_hours_NOT_latency": (received.dt.tz_localize(None) - end).dt.total_seconds() / 3600}).describe(percentiles=[.01, .5, .99]).round(2))
print("Calendar start-date counts (device wall dates; not a confirmed reporting timezone)")
display(start.dt.date.value_counts().sort_index().rename("drives").to_frame())

## Join validation and warning volume
First remove exact duplicates, then exclude ambiguous IDs. Use a validated many-to-one join to prevent fan-out. Unmatched warnings can reflect extraction boundaries, incomplete pagination, delayed drive reports, or missing source reports. Zero observed warnings is not proof of a failure or proof no warnings occurred.

In [ ]:
def unambiguous(df, key):
    unique = df.drop_duplicates()
    return unique.loc[unique[key].notna() & ~unique[key].duplicated(keep=False)].copy()

d = unambiguous(drives, "session_id")
w = unambiguous(warnings, "warning_id")
joined = w.merge(d[["session_id", "user_id", "started_at", "ended_at", "platform", "app_version"]], on="session_id", how="left", suffixes=("_warning", "_drive"), indicator=True, validate="many_to_one")
matched = joined["_merge"].eq("both")
user_comparable = matched & joined["user_id_warning"].notna() & joined["user_id_drive"].notna()
jstart, jend = parse_device(joined["started_at"]), parse_device(joined["ended_at"])
jtime = parse_device(joined["device_time"])
time_comparable = matched & jstart.notna() & jend.notna() & jtime.notna()
join_summary = {
    "unambiguous drives": len(d), "unambiguous warnings": len(w),
    "warnings with null session_id": int(w["session_id"].isna().sum()),
    "matched warning rows": int(matched.sum()), "unmatched warning rows": int((~matched).sum()),
    "matched warning pct": round(matched.mean() * 100, 2),
    "user comparable warning rows": int(user_comparable.sum()),
    "user mismatches": int((user_comparable & joined["user_id_warning"].ne(joined["user_id_drive"])).sum()),
    "time comparable warning rows": int(time_comparable.sum()),
    "warning outside drive interval": int((time_comparable & ((jtime < jstart) | (jtime > jend))).sum()),
}
display(pd.Series(join_summary, name="value").to_frame())
counts = w.groupby("session_id").size().rename("observed_warnings")
drive_warnings = d.join(counts, on="session_id")
drive_warnings["observed_warnings"] = drive_warnings["observed_warnings"].fillna(0).astype(int)
display(drive_warnings["observed_warnings"].describe(percentiles=[.5, .9, .95, .99]).to_frame())
display(drive_warnings["observed_warnings"].value_counts().sort_index().rename("sessions").to_frame().head(25))
display(drive_warnings.groupby(["platform", "app_version"]).agg(drives=("session_id", "size"), mean_observed_warnings=("observed_warnings", "mean"), median_observed_warnings=("observed_warnings", "median")))
print("Warning counts across all observed warning session IDs (including unmatched sessions)")
display(counts.describe(percentiles=[.5, .9, .95, .99]).to_frame())

## Evidence summary and candidate metrics
The next cell summarizes observed evidence. Candidate metrics below are proposals, not confirmed failure definitions.

In [ ]:
print(f"Coverage: {len(drives):,} drives and {len(warnings):,} warnings; pagination exhausted: drives={drive_extract['pagination_exhausted']}, warnings={warning_extract['pagination_exhausted']}.")
print(f"Join: {matched.mean():.1%} of unambiguous warning rows match an unambiguous extracted drive.")
print(f"Payload differences on repeated IDs (including receipt metadata): drives={identity(drives, 'session_id')['conflicting_ids']}, warnings={identity(warnings, 'warning_id')['conflicting_ids']}.")
print(f"Timestamp checks: {int((elapsed < 0).sum())} reversed intervals; {int((duration_delta.abs() > 1).sum())} duration mismatches over one second.")
print("A connection count above one records multiple connections; these fields alone do not establish a failed drive.")
print("No explicit crash, interrupted-drive outcome, or expected-warning count appears in the inspected schema. Missing reports are not observable as sessions in the drives endpoint.")

### Proposed small metric set
| Metric | Definition and purpose | Conditions / limits |
|---|---|---|
| Reported drive and driver volume | Distinct unambiguous session IDs and user IDs, by complete reporting week, platform, and version | Measures reported activity, not all attempted drives. Establish timezone, receipt completeness, and late-arrival policy first. |
| Connection-count distribution | Median/p95 connections and share of eligible reported drives with more than one recorded connection, by platform/version | Descriptive connection behavior only; not a failure rate. Exclude missing/invalid counts and show denominator. Control for drive duration and device mix before comparing releases. |
| Observed warning delivery / reaction | Observed matched warnings per drive or driving hour; recorded actions by type as a share of observed warnings | Only after join and coverage checks; deduplicate IDs, validate durations, and show null actions separately. No expected-warning denominator means this cannot measure missed alerts. |

Pair metrics with data-quality checks: ID conflicts, unparseable times, inconsistent durations, join coverage, receipt coverage, and freshness. A definitive bad-drive/crash-free metric is not supported by these fields alone. Compare releases within platform and show sample sizes; associations do not establish causation. Do not use users' current plan to infer historical revenue loss.

**Before weekly findings:** resolve device timestamp semantics, inspect any flagged anomalies, obtain comparable complete cohorts for both endpoints, and validate what connection counts mean. This notebook intentionally builds no dbt models or dashboards. Clear outputs before sharing if source data must be omitted; cached raw records stay under ignored `data/`.

### Reading the updated evidence
Use the executed tables above for current row counts, coverage, duplicate differences, timestamp flags, and join coverage. The original filtered samples have been replaced with an all-history traversal, without a date filter. Receipt-only replays are distinguished from business-field conflicts. The join remains conservative: all repeated IDs are excluded, even when only receipt metadata differs.

Exhausting pagination removes the former sampling and lower-date restrictions. The fixed upper receipt cutoff can still exclude late reports, and unreported sessions remain unobservable. Keep these limitations when proposing weekly metrics.